#KPI Calculations


### KPI 1 Monthly revenue and percentage

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi.kpi_1_monthly_revenue AS

WITH monthly_revenue AS (
    SELECT
        YEAR(posting_date) AS year,
        MONTH(posting_date) AS month,
        SUM(revenue) AS total_revenue
    FROM gold.fact.fact_general_ledgerse f
    JOIN gold.dim.dim_account a
        ON f.account_id = a.account_id
    WHERE a.account_type = 'Revenue'
    GROUP BY YEAR(posting_date), MONTH(posting_date)
),

with_prev AS (
    SELECT
        year,
        month,
        total_revenue,
        LAG(total_revenue) OVER (ORDER BY year, month) AS prev_revenue
    FROM monthly_revenue
)

SELECT
    year,
    month,
    total_revenue,
    CASE 
        WHEN prev_revenue IS NULL OR prev_revenue = 0 THEN NULL
        ELSE (total_revenue - prev_revenue) / ABS(prev_revenue)
    END AS mom_growth_pct
FROM with_prev;

In [0]:
%sql
select * from gold.kpi.kpi_1_monthly_revenue

In [0]:
%sql
SELECT 
    SUM(debit_amount) AS total_debit,
    SUM(credit_amount) AS total_credit,
    sum(revenue)
FROM silver.core.general_ledgers;

### KPI 2 cost of sales

In [0]:
%sql
SELECT
    a.account_id,
    a.account_code,
    a.account_name,
    a.account_type,
    a.category,
    g.debit_amount, 
    g.credit_amount,
    g.Revenue
FROM silver.transformation.accounts a
JOIN silver.core.general_ledgers g
    ON a.account_id = g.account_id
WHERE a.account_type IN ('COGS', 'Contra COGS')


In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi.kpi_2_cost_of_sales AS

SELECT
    f.company_id,
    YEAR(f.posting_date) AS year,
    MONTH(f.posting_date) AS month,
    abs(SUM(revenue) )AS total_cost_of_sales
FROM gold.fact.fact_general_ledgerse f
JOIN gold.dim.dim_account a
    ON f.account_id = a.account_id
WHERE a.account_type IN ('COGS', 'Contra COGS')
GROUP BY
    YEAR(f.posting_date),
    MONTH(f.posting_date),
     f.company_id
    

In [0]:
%sql
select * from gold.kpi.kpi_2_cost_of_sales

### KPI 3 Gross marigin percentage

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi.kpi_3_gross_profit_margin AS

WITH revenue AS (
    SELECT
        YEAR(f.posting_date) AS year,
        MONTH(f.posting_date) AS month,
        SUM(revenue) * -1 AS revenue_pos
    FROM gold.fact.fact_general_ledgerse f
    JOIN gold.dim.dim_account a
        ON f.account_id = a.account_id
    WHERE a.account_type = 'Revenue'
    GROUP BY YEAR(f.posting_date), MONTH(f.posting_date)
),

cost AS (
    SELECT
        YEAR(f.posting_date) AS year,
        MONTH(f.posting_date) AS month,
        SUM(revenue) * -1 AS cost_pos
    FROM gold.fact.fact_general_ledgerse f
    JOIN gold.dim.dim_account a
        ON f.account_id = a.account_id
    WHERE a.account_type IN ('COGS', 'Contra COGS')
    GROUP BY YEAR(f.posting_date), MONTH(f.posting_date)
)

SELECT
    r.year,
    r.month,
    r.revenue_pos AS total_revenue,
    c.cost_pos AS total_cost,

    (r.revenue_pos - c.cost_pos) AS gross_profit,

    CASE 
        WHEN r.revenue_pos = 0 THEN NULL
        ELSE (r.revenue_pos - c.cost_pos) / r.revenue_pos
    END AS gross_profit_margin

FROM revenue r
LEFT JOIN cost c
    ON r.year = c.year AND r.month = c.month;

In [0]:
%sql
select * from gold.kpi.kpi_3_gross_profit_margin

### KPI 4 Operating Expense Breakdown

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi.kpi_4_operating_expense_breakdown AS

SELECT
    YEAR(f.posting_date) AS year,
    MONTH(f.posting_date) AS month,
    f.company_id,
    a.category,
    SUM(revenue) * -1 AS total_expense
FROM gold.fact.fact_general_ledgerse f
JOIN gold.dim.dim_account a
    ON f.account_id = a.account_id
WHERE a.category = 'Operating Expense'
GROUP BY
    YEAR(f.posting_date),
    MONTH(f.posting_date),
    f.company_id,
    a.category;

In [0]:
%sql
select * from gold.kpi.kpi_4_operating_expense_breakdown

### KPI 5  Average Compensation

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi.kpi_5_avg_compensation AS

SELECT
    p.company_id,
    e.position,
    AVG(p.total_compensation) AS avg_compensation
FROM gold.fact.fact_payroll p
JOIN gold.dim.dim_employee e
    ON p.employee_id = e.employee_id
GROUP BY
    p.company_id,
    e.position;

In [0]:
%sql
select * from gold.kpi.kpi_5_avg_compensation

### KPI 6 net profit 

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi.kpi_6_net_profit AS

WITH revenue AS (
    SELECT
        YEAR(posting_date) AS year,
        MONTH(posting_date) AS month,
        SUM(revenue) * -1 AS revenue_pos
    FROM gold.fact.fact_general_ledgerse f
    JOIN gold.dim.dim_account a
        ON f.account_id = a.account_id
    WHERE a.account_type = 'Revenue'
    GROUP BY YEAR(posting_date), MONTH(posting_date)
),

cogs AS (
    SELECT
        YEAR(posting_date) AS year,
        MONTH(posting_date) AS month,
        SUM(revenue) * -1 AS cogs_pos
    FROM gold.fact.fact_general_ledgerse f
    JOIN gold.dim.dim_account a
        ON f.account_id = a.account_id
    WHERE a.account_type IN ('COGS','Contra COGS')
    GROUP BY YEAR(posting_date), MONTH(posting_date)
),

expense AS (
    SELECT
        YEAR(posting_date) AS year,
        MONTH(posting_date) AS month,
        SUM(revenue) * -1 AS expense_pos
    FROM gold.fact.fact_general_ledgerse f
    JOIN gold.dim.dim_account a
        ON f.account_id = a.account_id
    WHERE a.account_type = 'Expense'
    GROUP BY YEAR(posting_date), MONTH(posting_date)
)

SELECT
    r.year,
    r.month,
    r.revenue_pos,
    c.cogs_pos,
    e.expense_pos,

    (r.revenue_pos - c.cogs_pos - e.expense_pos) AS net_profit

FROM revenue r
LEFT JOIN cogs c
    ON r.year = c.year AND r.month = c.month
LEFT JOIN expense e
    ON r.year = e.year AND r.month = e.month;


In [0]:
%sql
select * from gold.kpi.kpi_6_net_profit

### KPI 7 overtime and bonus

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi.kpi_7_overtime_bonus AS

SELECT
    department_id,

    SUM(bonus) AS total_bonus,
    SUM(overtime_pay) AS total_overtime,

    SUM(bonus + overtime_pay) AS total_variable_pay,

    CASE 
        WHEN SUM(gross_salary) = 0 THEN NULL
        ELSE SUM(overtime_pay) / SUM(gross_salary)
    END AS overtime_to_salary_ratio

FROM gold.fact.fact_payroll
GROUP BY department_id;

In [0]:
%sql
select * from gold.kpi.kpi_7_overtime_bonus;

### KPI 8 avg per dept

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi.kpi_8_cost_per_department AS

SELECT
    department_id,
    SUM(total_compensation) AS total_department_cost
FROM gold.fact.fact_payroll
GROUP BY department_id;

In [0]:
%sql
select * from gold.kpi.kpi_8_cost_per_department

### KPI 9 headcount distribution

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi.kpi_9_headcount AS

SELECT
    department_id,
    COUNT(employee_id) AS headcount
FROM gold.dim.dim_employee
WHERE termination_date = CURRENT_DATE()
GROUP BY department_id;

In [0]:
%sql
select * from gold.kpi.kpi_9_headcount

### kpi 10 payroll efficiency 

In [0]:
%sql
CREATE OR REPLACE TABLE gold.kpi.kpi_10_payroll_to_revenue AS

WITH payroll AS (
    SELECT
        YEAR(pay_date) AS year,
        MONTH(pay_date) AS month,
        company_id,
        SUM(total_compensation) AS payroll_cost
    FROM gold.fact.fact_payroll
    GROUP BY
        YEAR(pay_date),
        MONTH(pay_date),
        company_id
),

revenue AS (
    SELECT
        YEAR(f.posting_date) AS year,
        MONTH(f.posting_date) AS month,
        f.company_id,
        SUM(revenue) * -1 AS revenue_pos
    FROM gold.fact.fact_general_ledgerse f
    JOIN gold.dim.dim_account a
        ON f.account_id = a.account_id
    WHERE a.account_type = 'Revenue'
    GROUP BY
        YEAR(f.posting_date),
        MONTH(f.posting_date),
        f.company_id
)

SELECT
    p.year,
    p.month,
    p.company_id,
    p.payroll_cost,
    r.revenue_pos,

    CASE 
        WHEN r.revenue_pos = 0 THEN NULL
        ELSE p.payroll_cost / r.revenue_pos
    END AS payroll_to_revenue_ratio

FROM payroll p
JOIN revenue r
    ON p.year = r.year 
   AND p.month = r.month 
   AND p.company_id = r.company_id;

In [0]:
%sql
select * from gold.kpi.kpi_10_payroll_to_revenue